sec7C — numbers quoted in Sec. VII C (dissipative impurity) and the captions of Figs. `fig:impss`, `fig:sigma`
Re-derived from the shipped data of `fig_impss`, `fig_sigma` and of this directory (stability runs, shape of Sigma).

In [1]:
import sys
from pathlib import Path
import numpy as np
HERE = Path.cwd()
sys.path.insert(0, str(HERE.parent))
from common import io as IO
ST = IO.read_data(HERE.parent / "fig_impss" / "data")["steady.csv"]
SG = IO.read_data(HERE.parent / "fig_sigma" / "data")["summary.csv"]
Q = IO.read_data(HERE / "data"); STAB, SF = Q["stability.csv"], Q["sigma_features.csv"]
ROWS = []
def check(where, printed, s):
    ROWS.append((where, printed, s, s == printed))
one = lambda tab, q, **w: IO.select(tab, **w)[q][0]
pct = lambda v: f"{100 * v:.0f}\\%"
def nd(sys_, m):
    s = IO.select(ST, system=sys_, method=m, ic="ic00"); o = np.argsort(s["gamma"])
    return s["gamma"][o], (s["n_up"] + s["n_dn"])[o], (s["D"] / (s["n_up"] * s["n_dn"]))[o]

In [2]:
# strongly damped model (c3)
g, ne, DOe = nd("c3", "exact"); _, nd_, DOd = nd("c3", "DSL"); _, _, DOt = nd("c3", "TPPX")
gp = g > 0
gm = g[gp][np.argmin(ne[gp])]
check("c3 exact n_d: Zeno minimum between gamma = 3 and 4 (gamma of the smallest n_d)", "3--4", "3--4" if gm in (3.0, 4.0) else f"{gm:g}")
check("c3 exact D/O at gamma = 3", "0.35", f"{DOe[g == 3.0][0]:.2f}")
check("c3 exact D/O at gamma = 24", "0.0082", f"{DOe[g == 24.0][0]:.4f}")
check("c3 DSL D/O at gamma = 24", "0.0084", f"{DOd[g == 24.0][0]:.4f}")
check("c3 DSL max |n_d - n_d,ex| over gamma", "0.005", "0.005" if np.abs(nd_ - ne).max() < 0.005 else f"{np.abs(nd_ - ne).max():.4f}")
check("c3 DSL mean |n_d - n_d,ex| over the 10 gamma > 0", "0.0017", f"{np.abs(nd_ - ne)[gp].mean():.4f}")
check("c3 TPPX D/O at gamma = 24", "0.26", f"{DOt[g == 24.0][0]:.2f}")
check("c3 2B n_d at gamma = 24 (saturates near 0.5)", "0.5", f"{nd('c3', '2B')[1][-1]:.1f}")
check("c3 mGW n_d at gamma = 24 (-> 1)", "1", f"{nd('c3', 'mGW')[1][-1]:.0f}")
s = IO.select(ST, system="c3", method="mGW")
check("c3 mGW runs with negative D (D_min < -1e-6)", "19 of its 33", f"{(s['D_min'] < -1e-6).sum()} of its {len(s['D_min'])}")
for m, p in (("HF", "0.28"), ("TPPX", "0.22")):
    s = IO.select(STAB, method=m, gamma=24.0)
    pert = IO.select(s, ic="pert"); pol = IO.select(s, ic="ic10")
    check(f"c3 {m}: |M_z| at t = 600 from M_z(0) = {pert['Mz0'][0]:.0e}, gamma = 24", p, f"{abs(pert['Mz_end'][0]):.2f}")
    check(f"c3 {m}: same as from the polarized start (difference)", "same", "same" if abs(abs(pert['Mz_end'][0]) - abs(pol['Mz_end'][0])) < 1e-3 else f"{abs(pert['Mz_end'][0]) - abs(pol['Mz_end'][0]):.1e}")
s = IO.select(STAB, ic="pert")
check("initial polarization of the stability runs", "$5\\times10^{-4}$", "$" + f"{s['Mz0'][0]:.0e}".replace("e-0", "\\times10^{-") + "}$")
t1e = lambda sys_, m: one(ST, "Mz_t1e", system=sys_, method=m, ic="ic10", gamma=24.0)
check("c3 DSL M_z 1/e time, gamma = 24", "2.11", f"{t1e('c3', 'DSL'):.2f}")
check("c3 exact M_z 1/e time, gamma = 24", "2.06", f"{t1e('c3', 'exact'):.2f}")
for m in ("2B", "mGW"):
    check(f"c3 {m} M_z 1/e time, gamma = 24 (relaxes too slowly: later than exact)", "too slowly",
          "too slowly" if t1e('c3', m) > t1e('c3', 'exact') + 0.5 else f"{t1e('c3', m):.2f}")
# weakly damped model (c3G0.2)
g, ne, _ = nd("c3G0.2", "exact"); gp = g > 0
check("c3G0.2 exact n_d: Zeno minimum", "4", f"{g[gp][np.argmin(ne[gp])]:g}")
check("c3G0.2 exact n_d at gamma = 24", "0.608", f"{ne[g == 24.0][0]:.3f}")
err = {m: np.abs(nd("c3G0.2", m)[1] - ne)[gp] for m in ("HF", "2B", "TPPX", "mGW", "DSL")}
best = sum(min(err, key=lambda m: err[m][k]) == "DSL" for k in range(gp.sum()))
check("c3G0.2: smallest mean |n_d - n_d,ex| over the 10 loss rates", "DSL", min(err, key=lambda m: err[m].mean()))
check("c3G0.2: (info, not quoted; text says most accurate over the range of gamma) loss rates at which the DSL has the smallest error", "9", f"{best}")
check("c3G0.2 DSL M_z 1/e time, gamma = 24", "1.97", f"{t1e('c3G0.2', 'DSL'):.2f}")
check("c3G0.2 exact M_z 1/e time, gamma = 24", "1.91", f"{t1e('c3G0.2', 'exact'):.2f}")
l5 = lambda m: one(ST, "Mz_last005", system="c3G0.2", method=m, ic="ic10", gamma=24.0)
check("c3G0.2 DSL: M_z last exceeds 0.05 at t", "18.3", f"{l5('DSL'):.1f}")
check("c3G0.2 exact: M_z last exceeds 0.05 at t", "10.4", f"{l5('exact'):.1f}")
check("c3G0.2 TPPX M_z 1/e time, gamma = 24", "63", f"{t1e('c3G0.2', 'TPPX'):.0f}")
check("c3G0.2 MF M_z at t' (stays polarized), gamma = 24", "0.28", f"{one(ST, 'Mz', system='c3G0.2', method='HF', ic='ic10', gamma=24.0):.2f}")

In [3]:
# self-energy (strongly damped, IC 00)
f = lambda m, g, q: one(SF, q, method=m, gamma=g)
check("exact -Im Sigma peak position (gamma = 4, 8, 16; near -1.7: within 0.1)", "-1.7",
      "-1.7" if all(abs(f('exact', g, 'peak_w') + 1.7) <= 0.1 for g in (4.0, 8.0, 16.0)) else
      ",".join(f"{f('exact', g, 'peak_w'):.2f}" for g in (4.0, 8.0, 16.0)))
r = [f("TPPX_0", g, "peak") / f("exact", g, "peak") - 1 for g in (4.0, 8.0, 16.0)]
check("TPPX_0 peak too high, smallest", "18\\%", pct(min(r))); check("TPPX_0 peak too high, largest", "31\\%", pct(max(r)))
check("TPPX_0 peak shifted to lower omega (all gamma)", "lower",
      "lower" if all(f("TPPX_0", g, "peak_w") < f("exact", g, "peak_w") for g in (4.0, 8.0, 16.0)) else "no")
check("DSL peak too high at gamma = 16", "23\\%", pct(f("DSL", 16.0, "peak") / f("exact", 16.0, "peak") - 1))
# features of the structure: exact extrema with |omega| <= 4.5 (15 of 16; excluded: the shallow high-frequency minimum of
# Re Sigma at omega = 4.95, gamma = 4, part of the approach to the large-|omega| limit)
SX = Q["sigma_extrema.csv"]; dw = SX["d_w"][(SX["method"] == "DSL") & (np.abs(SX["w_exact"]) <= 4.5)]
check("DSL feature positions: number of exact extrema with |omega| <= 4.5 (3 gamma)", "15", f"{len(dw)}")
check("DSL positions of the local extrema of -Im/Re Sigma, |omega| <= 4.5: mean |omega - omega_exact|", "0.08", f"{np.mean(dw):.2f}")
check("DSL positions of the local extrema of -Im/Re Sigma, |omega| <= 4.5: max |omega - omega_exact|", "0.24", f"{np.max(dw):.2f}")
for m, p in (("exact", "5.27"), ("DSL", "5.24")):
    check(f"gamma n_d,dn at gamma = 16 ({m}): large-|omega| limit of -Im Sigma", p, f"{one(SG, 'gamma_n_dn', method=m, gamma=16.0):.2f}")
    check(f"-Im Sigma_inf from G'(0) at gamma = 16 ({m})", p, f"{-one(SG, 'Sigma_inf_up_im', method=m, gamma=16.0):.2f}")
# positivity
s = IO.select(ST, method="DSL")
check("DSL D >= 0 in every impurity run (min D; gamma = 0 has no D)", "D >= 0", "D >= 0" if np.nanmin(s["D_min"]) > -1e-6 else f"{np.nanmin(s['D_min']):.1e}")
check("DSL eigenvalues of rho in [0, 1] (min, max)", "[0,1]",
      "[0,1]" if s["eig_min"].min() > -1e-6 and s["eig_max"].max() < 1 + 1e-6 else f"[{s['eig_min'].min():.1e},{s['eig_max'].max():.6f}]")
for sys_, p in (("c3", "$-0.082$"), ("c3G0.2", "$-0.42$")):
    s = IO.select(ST, method="DSL", system=sys_); pq = np.minimum(np.minimum(s["P_min"], s["Q_min"]), s["Gp_min"])
    check(f"{sys_}: DSL min P/Q/G' eigenvalue", p, f"${pq.min():.{3 if sys_ == 'c3' else 2}f}$")

In [4]:
w = max(len(r[0]) for r in ROWS)
print(f"{'quantity':{w}s}  {'paper':16s} {'new':16s} match")
for r in ROWS:
    print(f"{r[0]:{w}s}  {r[1]:16s} {r[2]:16s} {'yes' if r[3] else 'NO'}")
print(f"\n{sum(r[3] for r in ROWS)} of {len(ROWS)} match")

quantity                                                                                                                        paper            new              match
c3 exact n_d: Zeno minimum between gamma = 3 and 4 (gamma of the smallest n_d)                                                  3--4             3--4             yes
c3 exact D/O at gamma = 3                                                                                                       0.35             0.35             yes
c3 exact D/O at gamma = 24                                                                                                      0.0082           0.0082           yes
c3 DSL D/O at gamma = 24                                                                                                        0.0084           0.0084           yes
c3 DSL max |n_d - n_d,ex| over gamma                                                                                            0.005            0.005            yes
c3